# 01. 합성 데이터 생성 — 쇼핑몰 고객 지원팀 SFT 실습


> **시나리오 안내** — 이 실습은 가상의 이커머스 마켓플레이스 **쇼핑몰**(자체 PB 라인: 베이직 · 프리미엄 · 라이트 · 한정판)을 소재로 합니다. 실제 회사·상품·정책과 무관한 교육용 가상 시나리오입니다.

| 항목 | 내용 |
|------|------|
| 목적 | SFT(파인튜닝) 학습에 사용할 합성 대화 데이터셋 생성 |
| 교사 모델 | GPT-4o-mini (대형 모델 → 데이터 생성) |
| 학생 모델 | Qwen 3.5-2B (02 실습에서 이 데이터로 학습) |
| 런타임 | CPU (GPU 불필요) |
| 소요 시간 | ~25분 (전체 GPT 합성, 약 50개 문서 + 대화 생성) |
| 예상 비용 | GPT-4o-mini API 약 $0.30 (약 400원) |

**이 노트북에서 만드는 것:**
```
product-cs-sft-lab/
├── documents/          # 합성 내부 기술 문서 50개
└── sft-dataset/
    ├── train.jsonl     # SFT 학습용 대화 (80%)
    └── test.jsonl      # 평가용 대화 (20%, 학습에 절대 사용 금지)
```

> **사전 조건**: 00 노트북(Colab 기본 조작)을 먼저 완료하세요.

> 📝 **데이터 출처** — 모든 원본 문서는 GPT-4o-mini가 생성한 가상 콘텐츠입니다. 외부 사이트 크롤링·실제 기업 데이터 포함 없음 → **YouTube/GitHub 공개 시 저작권·크롤링·상표권 분쟁 위험 없음**.

---
## Phase 1: 환경 설정

In [1]:
# 필요 패키지 설치 — GPT 합성만 사용, 크롤링 라이브러리 불필요
!pip install --quiet openai
print("설치 완료 ✅")

설치 완료 ✅


In [5]:
import os
import json
import random
import time
from getpass import getpass
from pathlib import Path
from openai import OpenAI

## 원본 코드 - API 키 수동입력, 유출 위험 존재
"""
_key = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API Key: ")
if not _key:
    raise ValueError("API 키를 입력하세요.")
"""

## 보완 코드 - colab secrets에 OPENAI_API_KEY 등록 후 호출
from google.colab import userdata
_key = userdata.get('OPENAI_API_KEY')

client = OpenAI(api_key=_key)
TEACHER_MODEL = "gpt-4o-mini"
PROVIDER = "openai"

In [6]:
# 출력 디렉토리 생성
BASE_DIR = Path("product-cs-sft-lab")
DOCS_DIR = BASE_DIR / "documents"
SFT_DIR  = BASE_DIR / "sft-dataset"

for d in ["troubleshooting", "dev-guides", "qa-procedures", "customer-support", "meeting-notes"]:
    (DOCS_DIR / d).mkdir(parents=True, exist_ok=True)
SFT_DIR.mkdir(parents=True, exist_ok=True)

# 연결 확인
try:
    test_resp = client.chat.completions.create(
        model=TEACHER_MODEL,
        messages=[{"role": "user", "content": "ping"}],
        max_tokens=10,
    )
    print(f"✅ API 연결 성공: {PROVIDER} / {TEACHER_MODEL}")
except Exception as e:
    print(f"❌ API 연결 실패: {e}")

print(f"출력 경로: {BASE_DIR.resolve()}")

✅ API 연결 성공: openai / gpt-4o-mini
출력 경로: /content/product-cs-sft-lab


---
## Phase 2: 1단계 — 가상 원본 문서 생성 (전부 GPT 합성)

이번 실습은 **외부 사이트 크롤링 없이** 전적으로 **GPT-4o-mini가 생성한 가상 문서**로 SFT 데이터셋을 만듭니다.

**생성할 문서 카테고리:**

| 카테고리 | 예시 주제 | 목표 수량 |
|----------|-----------|----------|
| 서비스 트러블슈팅 | 배송 파손, 결제 오류, PB 상품 불량 | 15개 |
| 개발자/셀러 가이드 | 셀러 API, webhook, 정산 SDK | 10개 |
| QA 테스트 시나리오 | PB 내구성·방수, 부하·회귀 테스트 | 10개 |
| 고객 대응 매뉴얼 | 환불·교환·멤버십 FAQ | 10개 |
| 프로젝트 회의록 | 신제품 일정, 정책 개편 (가상) | 5개 |

> **주의**: 어떤 실제 기업의 내부 자료도 사용하지 않습니다. 모든 내용은 GPT가 합성한 가상의 쇼핑몰 사례입니다.

In [7]:
# 데이터 소스 — 전부 GPT 합성으로 생성할 가상 문서 주제 50개
# (외부 URL 파싱 없음 → 저작권/크롤링 약관/봇 차단 이슈 0건)

FAQ_SOURCES = {
    "troubleshooting": [
        "주문한 PB 무선 청소기가 배송 중 파손되어 도착한 경우 처리 절차",
        "배송 추적이 갱신되지 않는 상품에 대한 고객 응대 매뉴얼",
        "쇼핑몰 PB 노트북 가방의 지퍼 불량 케이스 응대 절차",
        "결제는 완료됐는데 주문 내역에 상품이 사라진 경우 시스템 조치",
        "쇼핑몰 PB 가습기 작동 불량 트러블슈팅 가이드",
        "쇼핑몰 PB 운동화 사이즈 표기 오류 발생 시 대응 절차",
        "쇼핑몰 PB 한정판 텀블러 단열 성능 미달 클레임 대응",
        "주소 변경이 늦어 오배송된 상품의 회수 절차",
        "다회용 쿠폰 중복 적용 오류 발생 시 시스템 조치",
        "쇼핑몰 PB 캐리어 바퀴 파손 보증 처리 절차",
        "포인트 적립 누락 케이스 응대 매뉴얼",
        "결제 후 환불까지 7일 초과 케이스 대응",
        "비밀번호 분실 후 본인 인증 실패 시 대응",
        "주문 취소 요청이 배송 시작 이후 들어왔을 때 처리",
        "선물 포장 옵션 누락 클레임 대응 절차",
    ],
    "dev-guides": [
        "쇼핑몰 셀러센터 Open API 인증 절차 v3 (OAuth 2.0)",
        "주문 상태 변경 이벤트 webhook 연동 가이드",
        "쇼핑몰 상품 이미지 업로드 SDK 사용법 (Python/Node.js)",
        "쇼핑몰 내부 추천 시스템 A/B 테스트 SDK 가이드",
        "쇼핑몰 PB 라이트 라인 카탈로그 API 사용 가이드",
        "쇼핑몰 멤버십 통합 SSO 연동 방법 (OIDC)",
        "쇼핑몰 검색 랭킹 알고리즘 파라미터 튜닝 가이드",
        "쇼핑몰 푸시 알림 SDK 셋업 가이드 (FCM 연동)",
        "쇼핑몰 결제 게이트웨이 PG 연동 가이드 (가상 PG)",
        "쇼핑몰 셀러 정산 API 인증 토큰 갱신 가이드",
    ],
    "qa-procedures": [
        "쇼핑몰 PB 노트북 가방 내구성 테스트 절차 (10만회 개폐)",
        "쇼핑몰 PB 운동화 방수 IP54 검증 테스트 방법",
        "쇼핑몰 PB 가습기 가습량/소음 측정 절차",
        "쇼핑몰 PB 텀블러 단열 성능 6시간 테스트",
        "쇼핑몰 검색 랭킹 알고리즘 회귀 테스트 시나리오",
        "쇼핑몰 모바일 앱 결제 페이지 부하 테스트 (TPS, p99 latency)",
        "쇼핑몰 PB 캐리어 낙하 테스트 시나리오 및 합격 기준",
        "쇼핑몰 PB 무선 이어폰 배터리 사이클 수명 테스트",
        "쇼핑몰 푸시 알림 발송 신뢰성 및 전송률 테스트 절차",
        "쇼핑몰 반품 자동화 워크플로우 E2E 회귀 테스트",
    ],
    "customer-support": [
        "쇼핑몰 환불 정책 (구매 후 7일 변심 / 14일 불량 케이스)",
        "쇼핑몰 PB 라인 보증 정책 (베이직 1년 / 프리미엄 2년)",
        "쇼핑몰 멤버십 등급별 혜택 안내 (실버/골드/플래티넘)",
        "쇼핑몰 포인트 적립·사용 정책 및 소멸 기간",
        "쇼핑몰 무료 배송 기준 (장바구니 금액별 / 멤버십별)",
        "쇼핑몰 PB 한정판 사전 예약 정책 및 취소 규정",
        "쇼핑몰 자주 묻는 질문 — 배송 카테고리",
        "쇼핑몰 자주 묻는 질문 — 결제 카테고리",
        "쇼핑몰 자주 묻는 질문 — 회원·로그인 카테고리",
        "쇼핑몰 PB 라이트 라인 교환 정책 (불량 7일 vs 변심 14일)",
    ],
    "meeting-notes": [
        "쇼핑몰 PB 한정판 신제품 출시 일정 검토 회의록 (가상)",
        "쇼핑몰 멤버십 등급제 개편 영향 분석 회의록 (가상)",
        "쇼핑몰 검색 알고리즘 LLM 도입 PoC 결과 공유 회의록 (가상)",
        "쇼핑몰 PB 운동화 신모델 카테고리 확장 회의록 (가상)",
        "쇼핑몰 신규 배송 권역 확대 로드맵 검토 회의록 (가상)",
    ],
}

total = sum(len(v) for v in FAQ_SOURCES.values())
print(f"총 가상 문서 주제: {total}개 (전부 GPT 합성)")
for cat, sources in FAQ_SOURCES.items():
    print(f"  🤖 {cat}: {len(sources)}개")


총 가상 문서 주제: 50개 (전부 GPT 합성)
  🤖 troubleshooting: 15개
  🤖 dev-guides: 10개
  🤖 qa-procedures: 10개
  🤖 customer-support: 10개
  🤖 meeting-notes: 5개


---
## Phase 3: 2단계 — GPT-4o-mini로 가상 문서 생성

**교사 모델(GPT-4o-mini)**에게 각 주제에 대한 가상 내부 기술 문서를 작성하게 합니다.

- 가상 쇼핑몰의 현실적인 운영 시나리오로 생성
- 어떤 실제 기업·상품·정책도 차용하지 않음
- 문서당 약 1,500~2,000자

In [8]:
def generate_document(topic: str, category: str) -> str:
    """GPT-4o-mini로 가상 쇼핑몰 내부 문서를 생성합니다."""
    instructions = {
        "troubleshooting":  "트러블슈팅 가이드 형식으로 작성하세요. 증상 → 원인 분석 → 단계별 조치 → 재현 확인 순서로 구성하세요.",
        "dev-guides":       "개발 가이드 형식으로 작성하세요. 사용 사례, 사전 준비, 코드 스니펫 또는 API 호출 흐름, 에러 코드 처리, 보안 주의사항을 포함하세요.",
        "qa-procedures":    "테스트 절차서 형식으로 작성하세요. 준비물, 테스트 단계, 합격 기준, 실패 시 대응을 포함하세요.",
        "customer-support": "고객 대응 매뉴얼 형식으로 작성하세요. 정책 요약, 케이스별 처리 절차, 예외 처리, 상담원 톤 가이드를 포함하세요.",
        "meeting-notes":    "회의록 형식으로 작성하세요. 참석자(직책만), 논의 내용, 결정 사항, 후속 과제(담당자는 직책으로만)를 포함하세요.",
    }
    prompt = f"""당신은 가상의 이커머스 마켓플레이스 '쇼핑몰'의 고객 지원팀 시니어 전문가입니다.
아래 주제에 대한 가상 내부 문서를 작성하세요.

주제: {topic}
형식: {instructions.get(category, "관련 내용을 상세하게 작성하세요.")}
분량: 1,500~2,000자
조건:
- '쇼핑몰'은 가상의 이커머스 회사입니다. 실제 기업명·실제 상품명·실제 인물명은 절대 포함하지 마세요.
- 개인정보(이름, 전화번호, 사번, 이메일 등)는 포함하지 마세요.
- 현실적이지만 명백히 가상의 PB 라인(베이직/프리미엄/라이트/한정판)과 정책 수치를 사용하세요.
- 한국어로 작성하세요."""

    response = client.chat.completions.create(
        model=TEACHER_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    return response.choices[0].message.content

In [9]:
# ★ 테스트: GPT가 가상 문서를 잘 생성하는지 확인
print("=== GPT 가상 문서 생성 테스트 ===")
test_topic = FAQ_SOURCES["troubleshooting"][0]
test_doc   = generate_document(test_topic, "troubleshooting")
print(f"주제: {test_topic}\n")
print(test_doc[:500] + "...")
print(f"\n글자 수: {len(test_doc)}자")

=== GPT 가상 문서 생성 테스트 ===
주제: 주문한 PB 무선 청소기가 배송 중 파손되어 도착한 경우 처리 절차

# 트러블슈팅 가이드: PB 무선 청소기 배송 중 파손 처리 절차

## 증상
고객이 주문한 PB 무선 청소기가 배송 중 파손되어 도착하였다고 신고함. 고객은 제품의 외관 손상, 부품 파손 또는 작동 불능 상태를 확인하였으며, 불만족스러운 구매 경험으로 인해 환불 또는 교환을 요청하고 있음.

## 원인 분석
1. **포장 불량**: 제품 포장이 배송 과정에서 적절히 이루어지지 않아, 충격이나 압력에 의해 파손되었을 가능성이 있음.
2. **운송 중 사고**: 물류 과정에서의 사고(낙하, 충돌 등)로 인해 제품이 손상되었을 수 있음.
3. **재고 관리 문제**: 출고 전에 제품이 이미 손상된 상태로 재고에 있었을 수 있음.
4. **배송업체의 문제**: 제품을 배송하는 과정에서 배송업체의 관리 소홀로 인해 파손이 발생했을 가능성 있음.

## 단계별 조치

### 1단계: 고객 상담
- 고객의 문의를 받고, 상품이 파손된 상태의 사진을 요청합니다.
- 고객의 주문번호 및 배송 정보...

글자 수: 1483자


In [10]:
# 전체 문서 생성 — 모두 GPT 합성
# ★ 예상 소요 시간: 약 50개 × 6초 = ~5분

def safe_filename(text: str, max_len: int = 30) -> str:
    """파일명으로 쓸 수 없는 문자를 제거합니다."""
    return text[:max_len].replace(" ", "_").replace("/", "_").replace("\\", "_").replace(":", "_")

generated_docs = []

for category, topics in FAQ_SOURCES.items():
    print(f"\n[{category}] {len(topics)}개")
    for i, topic in enumerate(topics):
        filename = f"{i+1:02d}_{safe_filename(topic)}.txt"
        filepath = DOCS_DIR / category / filename

        if filepath.exists():
            content = filepath.read_text(encoding="utf-8")
            print(f"  [{i+1}] 스킵 (이미 존재): {topic[:40]}")
        else:
            try:
                content = generate_document(topic, category)
                filepath.write_text(content, encoding="utf-8")
                print(f"  [{i+1}] 생성 완료: {topic[:40]} ({len(content)}자)")
                time.sleep(0.5)
            except Exception as e:
                print(f"  [{i+1}] 오류: {e}")
                continue

        generated_docs.append({
            "category": category,
            "topic":    topic,
            "content":  content,
            "path":     str(filepath),
        })

print(f"\n✅ 가상 문서 생성 완료: 총 {len(generated_docs)}개")


[troubleshooting] 15개
  [1] 생성 완료: 주문한 PB 무선 청소기가 배송 중 파손되어 도착한 경우 처리 절차 (1718자)
  [2] 생성 완료: 배송 추적이 갱신되지 않는 상품에 대한 고객 응대 매뉴얼 (1442자)
  [3] 생성 완료: 쇼핑몰 PB 노트북 가방의 지퍼 불량 케이스 응대 절차 (1743자)
  [4] 생성 완료: 결제는 완료됐는데 주문 내역에 상품이 사라진 경우 시스템 조치 (1814자)
  [5] 생성 완료: 쇼핑몰 PB 가습기 작동 불량 트러블슈팅 가이드 (1733자)
  [6] 생성 완료: 쇼핑몰 PB 운동화 사이즈 표기 오류 발생 시 대응 절차 (1784자)
  [7] 생성 완료: 쇼핑몰 PB 한정판 텀블러 단열 성능 미달 클레임 대응 (1934자)
  [8] 생성 완료: 주소 변경이 늦어 오배송된 상품의 회수 절차 (1717자)
  [9] 생성 완료: 다회용 쿠폰 중복 적용 오류 발생 시 시스템 조치 (1838자)
  [10] 생성 완료: 쇼핑몰 PB 캐리어 바퀴 파손 보증 처리 절차 (1563자)
  [11] 생성 완료: 포인트 적립 누락 케이스 응대 매뉴얼 (1695자)
  [12] 생성 완료: 결제 후 환불까지 7일 초과 케이스 대응 (1849자)
  [13] 생성 완료: 비밀번호 분실 후 본인 인증 실패 시 대응 (1800자)
  [14] 생성 완료: 주문 취소 요청이 배송 시작 이후 들어왔을 때 처리 (1687자)
  [15] 생성 완료: 선물 포장 옵션 누락 클레임 대응 절차 (1661자)

[dev-guides] 10개
  [1] 생성 완료: 쇼핑몰 셀러센터 Open API 인증 절차 v3 (OAuth 2.0) (2629자)
  [2] 생성 완료: 주문 상태 변경 이벤트 webhook 연동 가이드 (2387자)
  [3] 생성 완료: 쇼핑몰 상품 이미지 업로드 SDK 사용법 (Python/Node.js) (3364자)
  [4] 생성 완료: 쇼핑몰 내부 추천 시스템 A/B 테스트 SDK 가이드 (

---
## Phase 4: 3단계 — SFT 학습용 대화 생성

**SFT(Supervised Fine-Tuning)**는 모델에게 새로운 행동을 직접 가르치는 방법입니다.  
"이 도메인에서는 이 톤과 형식으로 답해라"는 패턴을 주입합니다.

앞에서 만든 문서를 바탕으로, **실제 고객과 상담원의 대화 형식**으로 데이터를 생성합니다.

**포인트:**
- 고객(user)은 **비전문가 말투**로 질문합니다. 매뉴얼 용어를 모릅니다.
- 상담원(assistant)은 **친절하고 단계별**로 안내합니다.
- 같은 문서에서 다양한 질문이 나올 수 있습니다.

### 생성될 JSONL 포맷 미리보기

각 대화는 아래와 같은 **JSON 한 덩어리**로 파일의 한 줄에 저장됩니다.

```json
{"messages": [
  {"role": "user", "content": "어제 산 PB 운동화가 사이즈가 너무 작아요. 환불이나 교환이 되나요?"},
  {"role": "assistant", "content": "안녕하세요, 쇼핑몰 고객지원입니다. 불편을 드려 죄송합니다. 변심에 의한 사이즈 교환은 수령 후 7일 이내 가능하며, 마이쇼핑몰 > 주문내역에서 \'교환 신청\'을 눌러주시면 회수 택배가 자동 예약됩니다. 회수 확인 후 새 사이즈로 재발송해드릴게요. 도움이 더 필요하시면 알려주세요!"}
]}
```

**왜 이 포맷인가?**

1. **OpenAI Chat Completions API 표준** — ChatGPT API가 정립한 이후 Anthropic(Claude), Qwen, LLaMA 등 거의 모든 LLM 제공사가 같은 구조를 채택. 다른 모델로 갈아타도 데이터 재사용 가능.
2. **`role`로 화자 구분** — 모델이 "user가 질문하면 assistant가 답한다"는 역할 구조를 학습하려면 `role` 필드가 필수. 나중에 `<|im_start|>user`, `<|im_start|>assistant` 같은 특수 토큰으로 변환됨.
3. **멀티턴 확장성** — 지금은 1턴(user-assistant 1쌍)이지만, 2턴·3턴 대화도 같은 구조로 확장 가능. 02 학습 노트북의 `SFTTrainer`와 `tokenizer.apply_chat_template()`이 이 배열을 그대로 처리.

**파일 저장 방식**: JSONL(JSON Lines) — 한 줄이 하나의 독립적인 JSON. 대화가 500개면 500줄짜리 파일이 됩니다. 줄 단위로 읽기·append·샤딩이 쉬워서 ML 학습 데이터의 사실상 표준 포맷입니다.

In [11]:
def generate_conversations(document_content: str, category: str, n: int = 10) -> list:
    """
    문서 내용을 바탕으로 SFT 학습용 대화 n개를 생성합니다.
    반환값: [{"messages": [{"role": ..., "content": ...}, ...]}, ...]
    """

    category_tone = {
        "troubleshooting":   "배송·결제·PB 상품 이상을 호소하는 고객과 쇼핑몰 고객 지원 상담원",
        "dev-guides":        "셀러 Open API·SDK·webhook 사용법을 묻는 셀러/개발자와 쇼핑몰 기술 지원 담당자",
        "qa-procedures":     "테스트 절차를 문의하는 QA 담당자와 쇼핑몰 플랫폼 테스트 엔지니어",
        "customer-support":  "환불·교환·멤버십·이용약관을 문의하는 고객과 쇼핑몰 고객지원 상담원",
        "meeting-notes":     "신규 정책·로드맵에 대해 질문하는 직원과 담당 팀원",
    }

    prompt = f"""아래 가상 쇼핑몰의 고객 지원팀 내부 문서를 바탕으로 고객/셀러 문의 대화 {n}개를 생성하세요.

[문서 내용]
{document_content[:3000]}

[대화 유형]
{category_tone[category]}

[조건]
- 모든 대화는 가상의 '쇼핑몰'을 배경으로 합니다. 실제 회사명·상품명은 절대 등장하지 마세요.
- user는 비전문가 말투로 질문합니다. 매뉴얼 용어를 모릅니다.
  나쁜 예: "주문 취소 후 환불 SLA 처리 기간을 확인 부탁드립니다"
  좋은 예: "어제 산 운동화를 환불하고 싶은데 언제쯤 돈이 들어오나요?"
- assistant는 쇼핑몰 고객지원 상담원처럼 친절하고 단계별로 안내합니다. 인사말("안녕하세요, 쇼핑몰 고객지원입니다")과 마무리 멘트를 포함합니다.
- 대화당 1~2턴(user-assistant 1~2회 교환)으로 구성합니다.
- 문서에 없는 내용은 지어내지 마세요.
- {n}개의 대화를 다양한 질문 유형으로 구성하세요.

[출력 형식] 반드시 아래 JSON 객체 형식으로만 반환하세요. 설명 없이 JSON만 출력하세요.
{{"conversations": [
  {{"messages": [
    {{"role": "user", "content": "질문 내용"}},
    {{"role": "assistant", "content": "답변 내용"}}
  ]}},
  ...
]}}"""

    response = client.chat.completions.create(
        model=TEACHER_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8,
        response_format={"type": "json_object"},
    )
    raw = response.choices[0].message.content
    data = json.loads(raw)

    # conversations 키 우선 탐색, 없으면 첫 번째 리스트 값 사용
    if isinstance(data, list):
        conversations = data
    elif "conversations" in data:
        conversations = data["conversations"]
    else:
        conversations = next((v for v in data.values() if isinstance(v, list)), [])

    # 각 아이템을 {"messages": [...]} 형식으로 정규화
    result = []
    for item in conversations:
        if isinstance(item, dict) and "messages" in item:
            result.append(item)
        elif isinstance(item, list):
            # 이미 메시지 배열인 경우
            result.append({"messages": item})
        elif isinstance(item, dict):
            # messages 키가 없고 다른 키로 메시지가 있는 경우
            turns = next((v for v in item.values() if isinstance(v, list)), None)
            if turns:
                result.append({"messages": turns})
    return result

In [12]:
# ★ 테스트: 첫 번째 문서로 대화 3개 생성해서 품질 확인
test_doc_entry = generated_docs[0]
print(f"문서: {test_doc_entry['topic']}")
print(f"카테고리: {test_doc_entry['category']}\n")

test_convs = generate_conversations(test_doc_entry["content"], test_doc_entry["category"], n=3)
print(f"생성된 대화 수: {len(test_convs)}개\n")

for i, conv in enumerate(test_convs, 1):
    print(f"--- 대화 {i} ---")
    for turn in conv.get("messages", []):
        role = "👤 고객" if turn["role"] == "user" else "💬 상담원"
        content = turn["content"]
        print(f"{role}: {content[:120]}..." if len(content) > 120 else f"{role}: {content}")
    print()

문서: 주문한 PB 무선 청소기가 배송 중 파손되어 도착한 경우 처리 절차
카테고리: troubleshooting

생성된 대화 수: 3개

--- 대화 1 ---
👤 고객: 안녕하세요, 제가 주문한 청소기가 배송될 때 깨져서 왔어요. 어떻게 해야 하나요?
💬 상담원: 안녕하세요, 쇼핑몰 고객지원입니다. 우선 불편을 드려 정말 죄송합니다. 고객님, 주문 번호와 제품을 받은 날짜, 그리고 파손된 상태의 사진을 보내주실 수 있으신가요? 확인 후 빠르게 처리해드릴게요.

--- 대화 2 ---
👤 고객: 청소기 교환하고 싶은데, 새로 오는 건 언제쯤 받을 수 있을까요?
💬 상담원: 안녕하세요, 쇼핑몰 고객지원입니다. 교환 요청을 도와드릴게요. 고객님께서 보내주신 정보를 바탕으로 새로운 제품을 준비 중입니다. 예상 배송일은 3일 후입니다. 추가로 궁금한 점이 있으시면 언제든지 말씀해 주세요!

--- 대화 3 ---
👤 고객: 환불을 요청하고 싶은데, 언제 돈이 들어오나요?
💬 상담원: 안녕하세요, 쇼핑몰 고객지원입니다. 환불 요청을 도와드리겠습니다. 환불 절차는 보통 3-5일 정도 소요됩니다. 환불 계좌 정보도 함께 알려주시면 더욱 빠르게 처리해드릴 수 있습니다.



In [13]:
# 전체 SFT 대화 배치 생성
# ★ 문서당 대화 10개 생성 → 50개 문서 × 10 = 500개 대화
# ★ 예상 소요 시간: 약 50개 × 8초 = ~7분

CONVS_PER_DOC = 10  # 문서당 생성할 대화 수

all_conversations = []
all_jsonl_path = SFT_DIR / "all.jsonl"

# 이미 생성된 파일이 있으면 로드
if all_jsonl_path.exists():
    with open(all_jsonl_path, encoding="utf-8") as f:
        all_conversations = [json.loads(line) for line in f if line.strip()]
    print(f"기존 파일 로드: {len(all_conversations)}개")

# 아직 생성되지 않은 문서에 대해서만 생성
already_done = len(all_conversations) // CONVS_PER_DOC
remaining_docs = generated_docs[already_done:]

print(f"전체 문서: {len(generated_docs)}개 | 완료: {already_done}개 | 남은: {len(remaining_docs)}개")

with open(all_jsonl_path, "a", encoding="utf-8") as f:
    for i, doc in enumerate(remaining_docs, already_done + 1):
        try:
            convs = generate_conversations(doc["content"], doc["category"], n=CONVS_PER_DOC)
            for conv in convs:
                f.write(json.dumps(conv, ensure_ascii=False) + "\n")
                all_conversations.append(conv)
            print(f"[{i}/{len(generated_docs)}] {doc['topic'][:40]}... → {len(convs)}개 생성")
            time.sleep(1)
        except Exception as e:
            print(f"[{i}/{len(generated_docs)}] 오류: {e}")

print(f"\n✅ 전체 대화 생성 완료: {len(all_conversations)}개")
print(f"저장 위치: {all_jsonl_path}")

전체 문서: 50개 | 완료: 0개 | 남은: 50개
[1/50] 주문한 PB 무선 청소기가 배송 중 파손되어 도착한 경우 처리 절차... → 10개 생성
[2/50] 배송 추적이 갱신되지 않는 상품에 대한 고객 응대 매뉴얼... → 10개 생성
[3/50] 쇼핑몰 PB 노트북 가방의 지퍼 불량 케이스 응대 절차... → 10개 생성
[4/50] 결제는 완료됐는데 주문 내역에 상품이 사라진 경우 시스템 조치... → 10개 생성
[5/50] 쇼핑몰 PB 가습기 작동 불량 트러블슈팅 가이드... → 10개 생성
[6/50] 쇼핑몰 PB 운동화 사이즈 표기 오류 발생 시 대응 절차... → 10개 생성
[7/50] 쇼핑몰 PB 한정판 텀블러 단열 성능 미달 클레임 대응... → 10개 생성
[8/50] 주소 변경이 늦어 오배송된 상품의 회수 절차... → 10개 생성
[9/50] 다회용 쿠폰 중복 적용 오류 발생 시 시스템 조치... → 10개 생성
[10/50] 쇼핑몰 PB 캐리어 바퀴 파손 보증 처리 절차... → 10개 생성
[11/50] 포인트 적립 누락 케이스 응대 매뉴얼... → 10개 생성
[12/50] 결제 후 환불까지 7일 초과 케이스 대응... → 10개 생성
[13/50] 비밀번호 분실 후 본인 인증 실패 시 대응... → 10개 생성
[14/50] 주문 취소 요청이 배송 시작 이후 들어왔을 때 처리... → 10개 생성
[15/50] 선물 포장 옵션 누락 클레임 대응 절차... → 10개 생성
[16/50] 쇼핑몰 셀러센터 Open API 인증 절차 v3 (OAuth 2.0)... → 10개 생성
[17/50] 주문 상태 변경 이벤트 webhook 연동 가이드... → 10개 생성
[18/50] 쇼핑몰 상품 이미지 업로드 SDK 사용법 (Python/Node.js)... → 10개 생성
[19/50] 쇼핑몰 내부 추천 시스템 A/B 테스트 SDK 가이드... → 10개 생성
[20/50] 쇼핑몰 PB 라이트 라인 카탈로그 API 사용 가이드.

---
## Phase 5: 데이터 분리 — train / test

SFT 데이터를 **반드시 train/test로 분리**합니다.

학습에 쓴 질문으로 평가하면 시험 문제를 미리 본 것과 같습니다.  
`test.jsonl`의 질문은 학습 과정에서 **한 번도 보지 못한** 질문이어야 합니다.

```
전체 (500~750개)
  ├── train.jsonl  80~90% — LoRA SFT 학습에 사용
  └── test.jsonl   10~20% — 평가에만 사용 (학습에 절대 사용 금지)
```

> **분리 시점**: 생성 직후, 학습 시작 전에 분리합니다.

In [14]:
# train / test 분리
TEST_RATIO = 0.2   # 20%를 test로
RANDOM_SEED = 42   # 재현 가능한 분리를 위한 시드

random.seed(RANDOM_SEED)
data = all_conversations.copy()
random.shuffle(data)

split_idx  = int(len(data) * (1 - TEST_RATIO))
train_data = data[:split_idx]
test_data  = data[split_idx:]

# 저장
train_path = SFT_DIR / "train.jsonl"
test_path  = SFT_DIR / "test.jsonl"

with open(train_path, "w", encoding="utf-8") as f:
    for item in train_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(test_path, "w", encoding="utf-8") as f:
    for item in test_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"✅ 데이터 분리 완료")
print(f"  train.jsonl: {len(train_data)}개 ({len(train_data)/len(data)*100:.0f}%)")
print(f"  test.jsonl:  {len(test_data)}개 ({len(test_data)/len(data)*100:.0f}%)")

✅ 데이터 분리 완료
  train.jsonl: 399개 (80%)
  test.jsonl:  100개 (20%)


In [15]:
# 결과 검증 및 샘플 확인
print("=" * 60)
print("최종 데이터셋 요약")
print("=" * 60)
print(f"문서:       {len(generated_docs)}개")
print(f"전체 대화:  {len(all_conversations)}개")
print(f"train:      {len(train_data)}개")
print(f"test:       {len(test_data)}개")
print()

def print_sample(label: str, sample: dict):
    print(f"[{label}]")
    for turn in sample.get("messages", []):
        role = "👤 고객" if turn["role"] == "user" else "💬 상담원"
        content = turn["content"]
        print(f"{role}: {content[:150]}..." if len(content) > 150 else f"{role}: {content}")
    print()

print_sample("train 샘플", train_data[0])
print_sample("test 샘플",  test_data[0])

print("⚠️  test.jsonl은 02 학습 노트북에 절대 입력하지 마세요.")
print("    03 평가 노트북에서만 사용합니다.")

최종 데이터셋 요약
문서:       50개
전체 대화:  499개
train:      399개
test:       100개

[train 샘플]
👤 고객: 주문 취소하면 언제 환불이 되나요?
💬 상담원: 안녕하세요, 쇼핑몰 고객지원입니다. 주문 취소 후 환불 처리에는 일반적으로 3-5일이 소요됩니다. 웹훅으로 상태가 바뀌면 알림을 받으실 수 있습니다.

[test 샘플]
👤 고객: 어제 주문한 신발이 사라졌어요. 이건 어떻게 처리되나요?
💬 상담원: 안녕하세요, 쇼핑몰 고객지원입니다. 주문 번호를 알려주시면, 시스템에서 확인해보겠습니다. 제가 도와드릴게요.

⚠️  test.jsonl은 02 학습 노트북에 절대 입력하지 마세요.
    03 평가 노트북에서만 사용합니다.


---
## Phase 6 (옵션): Google Drive에 저장

Colab 세션이 끊기면 `/content/` 아래 파일이 모두 사라집니다.  
`train.jsonl` / `test.jsonl`을 Google Drive에 복사해두면 02 실습에서 바로 불러올 수 있습니다.

In [16]:
# (옵션) Google Drive에 train.jsonl / test.jsonl 저장
# Colab에서 실행하면 자동 마운트 + 저장, 로컬 Jupyter면 로컬 경로 안내
import shutil

try:
    from google.colab import drive
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    # ── Colab: Google Drive에 저장 ─────────────────────────────
    drive.mount('/content/drive')

    # (1) train.jsonl / test.jsonl 저장 (02·03 노트북이 사용)
    DRIVE_DIR = Path("/content/drive/MyDrive/product-cs-sft-lab/sft-dataset")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)

    for fname, fpath in [("train.jsonl", train_path), ("test.jsonl", test_path)]:
        dst = DRIVE_DIR / fname
        shutil.copy(fpath, dst)
        print(f"✅ 저장 완료: {dst}")

    # (2) documents/ 폴더도 함께 저장 (06 Agentic RAG가 벡터 검색 대상으로 사용)
    DOCS_LOCAL = Path("product-cs-sft-lab/documents")
    DOCS_DRIVE = Path("/content/drive/MyDrive/product-cs-sft-lab/documents")
    if DOCS_LOCAL.exists():
        if DOCS_DRIVE.exists():
            shutil.rmtree(DOCS_DRIVE)
        shutil.copytree(DOCS_LOCAL, DOCS_DRIVE)
        n_files = sum(1 for _ in DOCS_DRIVE.rglob("*.txt"))
        print(f"✅ 저장 완료: {DOCS_DRIVE} ({n_files}개 .txt 파일)")

    print(f"\nDrive 경로: {DRIVE_DIR.parent}")
    print("후속 노트북에서 아래 경로로 불러옵니다:")
    print(f"  train:     {DRIVE_DIR}/train.jsonl  (02 학습)")
    print(f"  test:      {DRIVE_DIR}/test.jsonl   (03 평가)")
    print(f"  documents: {DOCS_DRIVE}/  (06 Agentic RAG)")

else:
    # ── 로컬 Jupyter: Drive 마운트 불가 ───────────────────────
    print("⚠️  Colab 환경이 아닙니다 (로컬 Jupyter/macOS 등). Drive 마운트 건너뜀.\n")
    print("현재 로컬 파일 경로:")
    print(f"  train: {train_path.resolve()}")
    print(f"  test:  {test_path.resolve()}\n")
    print("02 실습을 Colab에서 진행하시려면 다음 중 하나를 선택하세요:\n")
    print("  [방법 A] 위 두 파일을 브라우저로 Google Drive의 아래 경로에 직접 업로드:")
    print("           MyDrive/product-cs-sft-lab/sft-dataset/")
    print("           → 02 노트북의 Phase 3 데이터 로드 셀이 Drive에서 자동 로드합니다.\n")
    print("  [방법 B] 02 노트북도 같은 로컬 환경에서 실행 (GPU 필요).")
    print("           → Phase 3 로드 셀의 로컬 경로 분기가 그대로 동작합니다.")

Mounted at /content/drive
✅ 저장 완료: /content/drive/MyDrive/product-cs-sft-lab/sft-dataset/train.jsonl
✅ 저장 완료: /content/drive/MyDrive/product-cs-sft-lab/sft-dataset/test.jsonl
✅ 저장 완료: /content/drive/MyDrive/product-cs-sft-lab/documents (50개 .txt 파일)

Drive 경로: /content/drive/MyDrive/product-cs-sft-lab
후속 노트북에서 아래 경로로 불러옵니다:
  train:     /content/drive/MyDrive/product-cs-sft-lab/sft-dataset/train.jsonl  (02 학습)
  test:      /content/drive/MyDrive/product-cs-sft-lab/sft-dataset/test.jsonl   (03 평가)
  documents: /content/drive/MyDrive/product-cs-sft-lab/documents/  (06 Agentic RAG)


---
## 완료

**전체 산출물**

- `product-cs-sft-lab/documents/` — 가상 쇼핑몰 합성 문서 50개 (전부 GPT 합성, 외부 데이터 0건)
- `sft-dataset/train.jsonl` — SFT 학습용 (80%)
- `sft-dataset/test.jsonl` — 평가용 (20%, 학습 절대 금지)
- (필수) Drive 백업

**다음 실습**: `02_sft_train.ipynb` — train.jsonl로 Qwen 3.5-2B LoRA 학습 → "쇼핑몰 고객지원 상담원" 톤·형식 학습

> ✅ **공개 배포 가능** — 본 노트북은 외부 사이트 크롤링 없이 GPT 합성만 사용하므로 GitHub/YouTube 공개 시 저작권·크롤링 약관·상표권 분쟁 위험이 없습니다.